In [324]:
# Nivel 1 do desafio

In [325]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as map
import os
import json


In [326]:
#Lietura do arquivo josn
with open("../data/dados_nivel_1.json", "r", encoding="utf-8") as f:
    dados =  json.load(f)

#a taxa de cambio fixa 
taxa_cambio = dados["taxa_cambio_usd_brl"]

#convertando para dataframe
df = pd.DataFrame(dados['operacoes'])

print(taxa_cambio)
df.head()

5.4


,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
0,OP-0001,CLI-A-1,2026-03-09,18100,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
1,OP-0002,CLI-A-1,2026-03-09,17300,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
2,OP-0003,CLI-A-1,2026-03-09,18800,BRL,ted,transferencia_enviada,Beta Servicos ME,
3,OP-0004,CLI-A-1,2026-03-21,3300,BRL,boleto,pagamento,Gama Distribuidora,
4,OP-0005,CLI-A-2,2026-03-14,25900,BRL,ted,transferencia_enviada,Delta Transportes,


In [327]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   id           20 non-null     str  
 1   cliente_id   20 non-null     str  
 2   data         19 non-null     str  
 3   valor        20 non-null     int64
 4   moeda        20 non-null     str  
 5   canal        20 non-null     str  
 6   tipo         20 non-null     str  
 7   contraparte  20 non-null     str  
 8   observacao   20 non-null     str  
dtypes: int64(1), str(8)
memory usage: 2.9 KB


## Parte A - EDA

In [328]:
df

,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
0,OP-0001,CLI-A-1,2026-03-09,18100,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
1,OP-0002,CLI-A-1,2026-03-09,17300,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
2,OP-0003,CLI-A-1,2026-03-09,18800,BRL,ted,transferencia_enviada,Beta Servicos ME,
3,OP-0004,CLI-A-1,2026-03-21,3300,BRL,boleto,pagamento,Gama Distribuidora,
4,OP-0005,CLI-A-2,2026-03-14,25900,BRL,ted,transferencia_enviada,Delta Transportes,
5,OP-0006,CLI-A-2,2026-03-14,27000,BRL,ted,transferencia_enviada,Delta Transportes,
6,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,
7,OP-0008,CLI-A-3,2026-03-05,15200,BRL,pix,transferencia_enviada,Epsilon Consultoria,
8,OP-0009,CLI-A-3,2026-03-05,16100,BRL,pix,transferencia_enviada,Zeta Importacao,
9,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,


In [329]:
#passando data para o formato pandas
df["data"] = pd.to_datetime(df["data"])

In [330]:
#Encontrando e visualiando dados duplicados
df.duplicated()
df[df.duplicated()]

,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
9,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,


In [331]:
df = df.drop_duplicates()

In [332]:
df

,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
0,OP-0001,CLI-A-1,2026-03-09,18100,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
1,OP-0002,CLI-A-1,2026-03-09,17300,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
2,OP-0003,CLI-A-1,2026-03-09,18800,BRL,ted,transferencia_enviada,Beta Servicos ME,
3,OP-0004,CLI-A-1,2026-03-21,3300,BRL,boleto,pagamento,Gama Distribuidora,
4,OP-0005,CLI-A-2,2026-03-14,25900,BRL,ted,transferencia_enviada,Delta Transportes,
5,OP-0006,CLI-A-2,2026-03-14,27000,BRL,ted,transferencia_enviada,Delta Transportes,
6,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,
7,OP-0008,CLI-A-3,2026-03-05,15200,BRL,pix,transferencia_enviada,Epsilon Consultoria,
8,OP-0009,CLI-A-3,2026-03-05,16100,BRL,pix,transferencia_enviada,Zeta Importacao,
10,OP-0010,CLI-A-4,2026-03-03,3800,BRL,cartao,pagamento,Alfa Comercio LTDA,


A operação id="OP-0013" fez uma transferencia de USD12k, ortanto precisaremos fazer a cnoversão entre moedas.  

In [333]:
valor_dol = df.loc[13]["valor"]
print(valor_dol)
valor_brl = round(taxa_cambio*valor_dol, 2)
print(valor_brl)
df.loc[13,"valor"] = valor_brl
df.loc[13, "moeda"] = "BRL"
df.loc[13]

12000
64800.0


id                            OP-0013
cliente_id                    CLI-A-4
data              2026-03-24 00:00:00
valor                           64800
moeda                             BRL
canal                             ted
tipo           transferencia_recebida
contraparte           Zeta Importacao
observacao      remessa internacional
Name: 13, dtype: object

Uma das linha não possui todas as informações necessárias -  sem informação da data da transação.

In [334]:
df[df["data"].isna()]

,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
17,OP-0017,CLI-A-5,NaT,4300,BRL,especie,deposito,Gama Distribuidora,data nao capturada pelo sistema


A quantidade de transações por clientes

In [335]:
df['cliente_id'].value_counts()

cliente_id
CLI-A-1    4
CLI-A-4    4
CLI-A-5    4
CLI-A-3    3
CLI-A-2    2
CLI-A-6    2
Name: count, dtype: int64

In [336]:
df.sort_values(['cliente_id','valor'])

,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
3,OP-0004,CLI-A-1,2026-03-21,3300,BRL,boleto,pagamento,Gama Distribuidora,
1,OP-0002,CLI-A-1,2026-03-09,17300,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
0,OP-0001,CLI-A-1,2026-03-09,18100,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
2,OP-0003,CLI-A-1,2026-03-09,18800,BRL,ted,transferencia_enviada,Beta Servicos ME,
4,OP-0005,CLI-A-2,2026-03-14,25900,BRL,ted,transferencia_enviada,Delta Transportes,
5,OP-0006,CLI-A-2,2026-03-14,27000,BRL,ted,transferencia_enviada,Delta Transportes,
7,OP-0008,CLI-A-3,2026-03-05,15200,BRL,pix,transferencia_enviada,Epsilon Consultoria,
8,OP-0009,CLI-A-3,2026-03-05,16100,BRL,pix,transferencia_enviada,Zeta Importacao,
6,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,
10,OP-0010,CLI-A-4,2026-03-03,3800,BRL,cartao,pagamento,Alfa Comercio LTDA,


In [348]:
df.groupby('canal')['valor'].agg(
    ['count', 'sum', 'mean', 'min', 'max']
)

,count,sum,mean,min,max
canal,,,,,
boleto,3,11100,3700.0,2700,5100
cartao,2,5200,2600.0,1400,3800
especie,1,4300,4300.0,4300,4300
pix,8,101400,12675.0,2900,18100
ted,5,143500,28700.0,7000,64800


In [347]:
df.groupby('cliente_id')['valor'].agg(
    ['count', 'sum', 'mean', 'median', 'min', 'max']
)

,count,sum,mean,median,min,max
cliente_id,,,,,,
CLI-A-1,4,57500,14375.000000,17700.0,3300,18800
CLI-A-2,2,52900,26450.000000,26450.0,25900,27000
CLI-A-3,3,48500,16166.666667,16100.0,15200,17200
CLI-A-4,4,79500,19875.000000,5450.0,3800,64800
CLI-A-5,4,16900,4225.000000,3600.0,2700,7000
CLI-A-6,2,10200,5100.000000,5100.0,1400,8800


In [337]:
#Agrupando por valor total de transações por cliente
df.groupby("cliente_id")["valor"].sum()



cliente_id
CLI-A-1    57500
CLI-A-2    52900
CLI-A-3    48500
CLI-A-4    79500
CLI-A-5    16900
CLI-A-6    10200
Name: valor, dtype: int64

Analisando os canais que foram utilizadas pelos clientes

In [338]:
df.groupby("cliente_id")['canal'].value_counts()

cliente_id  canal  
CLI-A-1     pix        2
            ted        1
            boleto     1
CLI-A-2     ted        2
CLI-A-3     pix        3
CLI-A-4     cartao     1
            boleto     1
            pix        1
            ted        1
CLI-A-5     pix        1
            ted        1
            boleto     1
            especie    1
CLI-A-6     pix        1
            cartao     1
Name: count, dtype: int64

Os clientes costumam escolher o pix como forma de transação

Como data é um atributo importante para realizar a plicação das regras, então farei uma copia do df original e apagarei a linha que não possui essa informação.

In [339]:
df_cpy = df.dropna(subset=["data"])


In [340]:
fracionamento = df_cpy.groupby(['cliente_id', 'data']).agg(
    qtd_datas=('data','count'),
    soma_valores=('valor', 'sum')
)

fracionamento['fracionamento'] = (fracionamento['qtd_datas'] >=3) & (fracionamento['soma_valores'] >= 50000)
print(fracionamento)

                       qtd_datas  soma_valores  fracionamento
cliente_id data                                              
CLI-A-1    2026-03-09          3         54200           True
           2026-03-21          1          3300          False
CLI-A-2    2026-03-14          2         52900          False
CLI-A-3    2026-03-05          3         48500          False
CLI-A-4    2026-03-03          1          3800          False
           2026-03-11          1          5100          False
           2026-03-18          1          5800          False
           2026-03-24          1         64800          False
CLI-A-5    2026-03-07          1          2900          False
           2026-03-16          1          7000          False
           2026-03-26          1          2700          False
CLI-A-6    2026-03-12          1          8800          False
           2026-03-28          1          1400          False


Os clientes cujos id são ('CLI-A-1', 'CLI-A-3) foram os únicos a realizarem 3 transações num mesmo dia, porém somente o primeiro ('CLI-A-1') ultrapassou o limite de 50k.

In [341]:
df_cpy =  df_cpy.merge(fracionamento[['fracionamento']], on=['cliente_id', 'data'], how='left')
df_cpy

,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao,fracionamento
0,OP-0001,CLI-A-1,2026-03-09,18100,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,,True
1,OP-0002,CLI-A-1,2026-03-09,17300,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,,True
2,OP-0003,CLI-A-1,2026-03-09,18800,BRL,ted,transferencia_enviada,Beta Servicos ME,,True
3,OP-0004,CLI-A-1,2026-03-21,3300,BRL,boleto,pagamento,Gama Distribuidora,,False
4,OP-0005,CLI-A-2,2026-03-14,25900,BRL,ted,transferencia_enviada,Delta Transportes,,False
5,OP-0006,CLI-A-2,2026-03-14,27000,BRL,ted,transferencia_enviada,Delta Transportes,,False
6,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,,False
7,OP-0008,CLI-A-3,2026-03-05,15200,BRL,pix,transferencia_enviada,Epsilon Consultoria,,False
8,OP-0009,CLI-A-3,2026-03-05,16100,BRL,pix,transferencia_enviada,Zeta Importacao,,False
9,OP-0010,CLI-A-4,2026-03-03,3800,BRL,cartao,pagamento,Alfa Comercio LTDA,,False


In [342]:
df_cpy['qtd_operacoes'] = (
    df_cpy.groupby('cliente_id')['cliente_id']
    .transform('count')
)

df_cpy['mediana_cliente'] = (
    df_cpy.groupby('cliente_id')['valor']
    .transform('median')
)


df_cpy['sup'] = (
    (df_cpy['qtd_operacoes'] >= 4) &
    (df_cpy['valor'] >= df_cpy['mediana_cliente'] * 5)
)

df_cpy

,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao,fracionamento,qtd_operacoes,mediana_cliente,sup
0,OP-0001,CLI-A-1,2026-03-09,18100,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,,True,4,17700.0,False
1,OP-0002,CLI-A-1,2026-03-09,17300,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,,True,4,17700.0,False
2,OP-0003,CLI-A-1,2026-03-09,18800,BRL,ted,transferencia_enviada,Beta Servicos ME,,True,4,17700.0,False
3,OP-0004,CLI-A-1,2026-03-21,3300,BRL,boleto,pagamento,Gama Distribuidora,,False,4,17700.0,False
4,OP-0005,CLI-A-2,2026-03-14,25900,BRL,ted,transferencia_enviada,Delta Transportes,,False,2,26450.0,False
5,OP-0006,CLI-A-2,2026-03-14,27000,BRL,ted,transferencia_enviada,Delta Transportes,,False,2,26450.0,False
6,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,,False,3,16100.0,False
7,OP-0008,CLI-A-3,2026-03-05,15200,BRL,pix,transferencia_enviada,Epsilon Consultoria,,False,3,16100.0,False
8,OP-0009,CLI-A-3,2026-03-05,16100,BRL,pix,transferencia_enviada,Zeta Importacao,,False,3,16100.0,False
9,OP-0010,CLI-A-4,2026-03-03,3800,BRL,cartao,pagamento,Alfa Comercio LTDA,,False,4,5450.0,False


Ao analisarmos o dataframe, nota-se que a regra funcionou, pois no id = OP-0013:

qtd_operações = 4

Valor = 64800

Mediana = 5450 -> 5*Mediana = 27250 

logo, 64,8K é um valor atipico segundo as regras

## Parte B - Análise com LLM

In [351]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

ModuleNotFoundError: No module named 'torch'